In [1]:
from imagededup.methods import PHash
distance_threshold = 20  # Adjust this threshold based on your needs

def find_gallery_duplicates(gallery_path, distance_threshold):
  phasher = PHash()

  #print("Step 1: Generating encodings for gallery images...")
  # 1. Generate hashes for all images in the directory
  encodings = phasher.encode_images(image_dir=gallery_path)

  #print("Step 2: Finding duplicates using encodings...")
  # 2. Find duplicates using the generated encoding map
  duplicates_dict = phasher.find_duplicates(
      encoding_map=encodings,
      max_distance_threshold=distance_threshold,
  )

  return duplicates_dict


# Example usage:
dupes = find_gallery_duplicates(
    "/home/zuzia/Desktop/studia/year2/creativity/photo_book/photos",
    distance_threshold
)
duplicates = []
for img, duplicate_list in dupes.items():
  if duplicate_list:
    print(f"{img} has duplicates: {duplicate_list}")
    #make a list of all duplicates
    # Sort to ensure consistent ordering for deduplication
    group = sorted([img] + duplicate_list)
    group_str = ", ".join(group)
    duplicates.append(group_str)

#make a pandas dataframe with the duplicates
import pandas as pd
# Deduplicate by converting to set then back to list
unique_duplicates = list(set(duplicates))
duplicate_list = pd.DataFrame(unique_duplicates, columns=["Duplicate Images"])
#have only one row for each duplicate image, and the rest of the duplicates in the same row, separated by commas
# df = df.groupby("Duplicate Images").agg(lambda x: ', '.join(x)).reset_index()
print(duplicate_list)

2026-11-17 19:24:19,411: INFO Start: Calculating hashes...
100%|██████████| 24/24 [00:03<00:00,  7.65it/s]
2026-11-17 19:24:22,802: INFO End: Calculating hashes!
/home/zuzia/miniforge3/lib/python3.12/site-packages/imagededup/methods/hashing.py:317: RuntimeWarning: Parameter num_enc_workers has no effect since encodings are already provided
  warnings.warn('Parameter num_enc_workers has no effect since encodings are already provided', RuntimeWarning)
2026-11-17 19:24:22,807: INFO Start: Evaluating hamming distances for getting duplicates
2026-11-17 19:24:22,810: INFO Start: Retrieving duplicates using Cython Brute force algorithm
100%|██████████| 24/24 [00:00<00:00, 206700.81it/s]
2026-11-17 19:24:22,990: INFO End: Retrieving duplicates using Cython Brute force algorithm
2026-11-17 19:24:22,991: INFO End: Evaluating hamming distances for getting duplicates


20170807_220103.jpg has duplicates: ['20170807_220058.jpg']
20161001_121535.jpg has duplicates: ['20161001_121519.jpg']
20161001_115640.jpg has duplicates: ['20190218_130242.jpg']
20170806_145240.jpg has duplicates: ['20170806_145238.jpg']
20170807_220058.jpg has duplicates: ['20170807_220103.jpg']
20190218_130243.jpg has duplicates: ['20190218_130242.jpg', 'Screenshot_2017-05-19-17-35-06.png']
20190218_130242.jpg has duplicates: ['20190218_130243.jpg', '20161001_115640.jpg']
Screenshot_2017-05-19-17-35-06.png has duplicates: ['20190218_130243.jpg']
20161001_121519.jpg has duplicates: ['20161001_121535.jpg']
20170806_145238.jpg has duplicates: ['20170806_145240.jpg']
                                    Duplicate Images
0  20161001_115640.jpg, 20190218_130242.jpg, 2019...
1           20170807_220058.jpg, 20170807_220103.jpg
2  20190218_130243.jpg, Screenshot_2017-05-19-17-...
3           20170806_145238.jpg, 20170806_145240.jpg
4           20161001_115640.jpg, 20190218_130242.jpg
5  201

In [15]:
import face_recognition


def are_eyes_closed(image_path):
  # Load the image
  image = face_recognition.load_image_file(image_path)

  # Get facial landmarks for all faces in the image
  face_landmarks_list = face_recognition.face_landmarks(image)

  if not face_landmarks_list:
    print("No face found.")
    return False  # No face found
  print("Face found.")

  for face_landmarks in face_landmarks_list:
    left_eye = face_landmarks.get("left_eye")
    right_eye = face_landmarks.get("right_eye")

    if not left_eye or not right_eye:
      continue

    # Simple heuristic: compute vertical-to-horizontal ratio of the eye coordinates
    def get_eye_openness(eye):
      # eye has 6 points: [p0(corner), p1(top), p2(top), p3(corner), p4(bottom), p5(bottom)]
      vertical_1 = ((eye[1][0] - eye[5][0]) ** 2 + (eye[1][1] - eye[5][1]) ** 2) ** 0.5
      vertical_2 = ((eye[2][0] - eye[4][0]) ** 2 + (eye[2][1] - eye[4][1]) ** 2) ** 0.5
      horizontal = ((eye[0][0] - eye[3][0]) ** 2 + (eye[0][1] - eye[3][1]) ** 2) ** 0.5
      return (vertical_1 + vertical_2) / (2.0 * horizontal)

    left_openness = get_eye_openness(left_eye)
    right_openness = get_eye_openness(right_eye)

    avg_openness = (left_openness + right_openness) / 2.0

    # Threshold for closed eyes (typically below ~0.20)
    if avg_openness < 0.20:
      print("Eyes are closed.")
      return True  # Eyes are closed
      
    print("Eyes are open.")
  return False

In [19]:
are_eyes_closed("/home/zuzia/Desktop/studia/year2/creativity/photo_book/photos/20161001_115547.jpg")

No face found.


False

This code have a problem for now and usually do not found the face. Need to look at something else 